# Fine-Tuning Llama-3.2-3B

## Installing trl (Transformers Reinforcement Library) and wandb (Weights and Biases)

In [2]:
# Run the line below if trl and wandb are not already installed:
#!pip install trl wandb

## Imports

In [4]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

## Setting up Constants and Hyper-Parameters for QLoRA and Fine-Tuning:

In [5]:
# Constants:
BASE_MODEL = 'meta-llama/Llama-3.2-3B'
PROJECT_NAME = 'price'
HF_USER = 'Shailya01'
LITE_MODE = True # Turning this to False will use The Full Dataset with 800k Training Points.
DATASET_NAME = f'{HF_USER}/items_prompt_lite' if LITE_MODE else f'{HF_USER}/items_prompt_full'
RUN_NAME = f'{datetime.now():%Y-%m-%d_%H.%M.%S}'

if LITE_MODE:
    RUN_NAME += '-lite'
PROJECT_RUN_NAME = f'{PROJECT_NAME}--{RUN_NAME}'
HUB_MODEL_NAME = f'{HF_USER}/{PROJECT_RUN_NAME}'

In [6]:
# Hyper-Parameters (Overall):
epochs = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

In [7]:
# Hyper-Parameters for QLoRA:
QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ['q_proj', 'v_proj', 'k_proj', 'o_proj']
MLP_LAYERS = ['gate_proj', 'up_proj', 'down_proj']
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

In [8]:
# Hyper-Parameters for Training:
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = 'paged_adamw_32bit'
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

In [11]:
# Hyper-Parameters for Tracking the Training on Weights and Biases:
LOG_TO_WANDB = True # Turn this to False if you do not want to log and track Training on Weights and Biases.
VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200

## Log in to Weights and Biases and Hugging Face:

In [12]:
# Hugging Face:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [13]:
# Weights and Biases:
wandb_api_key = os.getenv('WANDB_API_KEY')
wandb.login(key= wandb_api_key)

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\shail\_netrc
wandb: Currently logged in as: shailya-gandhi100 (shailya-gandhi100-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [14]:
# Configure Weights & Biases to record against our project:
os.environ['WANDB_PROJECT'] = PROJECT_NAME
os.environ['WANDB_LOG_MODEL'] = 'false'
os.environ['WANDB_WATCH'] = 'false'

## Loading Dataset:

In [16]:
dataset = load_dataset(path= DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))
test = dataset['test']